# PC-FNO3D réel NATIONAL (Tunisie, nœuds 1 km) — entraînement et test (Colab, A100)

Même méthode que le Grand Tunis v4 (34 canaux : ERA5 aux nœuds + forçage de l'instant, sortie u, v
= ERA5 + correction, w par continuité), sur le jeu national (carnet de génération, 3 parties sur le
Drive). Entrée de production : ERA5 10 m tous les 2 km + les 12 aéroports METAR. **Départ : les
poids du modèle Grand Tunis livré** (`decision_v4.json`), simple point de départ : l'échelle d'un
mode de Fourier est ici 10 fois plus grande.

**Environnement : GPU A100.** Reprise automatique : relancer **Tout exécuter** après une coupure
reprend à la dernière copie sur le Drive. La session s'arrête à la fin. Résultats :
`MyDrive/pcfno3d_national/runs/national_1km_v2/`.

## 1. Drive, code et données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")   # code et poids du Grand Tunis
NAT = Path("/content/drive/MyDrive/pcfno3d_national")
NAT.mkdir(parents=True, exist_ok=True)
JOURNAUX = NAT / "journaux"
JOURNAUX.mkdir(exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"place libre sur le Drive : {libre:.0f} Go (le jeu national en demande ~50)")

CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille nationale, grille_reelle.json, forcage ERA5 (une archive sur le Drive)
DON = Path("/content/donnees")
G = DON / "national"
if not (G / "forcage_era5.csv").exists():
    with zipfile.ZipFile(NAT / "national_donnees.zip") as zf:
        zf.extractall(DON)
print(sorted(p.name for p in G.iterdir()))

## 2. Cache (3 parties du Drive) et contrôle de la vérité

In [ ]:
LOC = Path("/content/national")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (NAT / f"cache_v2_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_v2_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(NAT / f"cache_v2_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(NAT / "jeu_v2.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())
lancer("controle", "python -u -m pcfno3d -c national_1km_colab check")

## 3. Mémoire du GPU

In [ ]:
import torch
from pcfno3d.config import Config
from pcfno3d.geometry import Geometry
from pcfno3d.model import build_model

def essai(recalcul: bool) -> float:
    cfg = Config.load("configs/national_1km_colab.yaml")
    cfg.model.recalcul_activations = recalcul
    geom = Geometry(cfg, device="cuda")
    m = build_model(cfg, geom.c_in).cuda().train()
    x = torch.randn(cfg.train.batch_size, geom.c_in, *geom.shape, device="cuda")
    torch.cuda.reset_peak_memory_stats()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        m(x).float().pow(2).mean().backward()
    pic = torch.cuda.max_memory_allocated() / 1e9
    del m, x
    torch.cuda.empty_cache()
    return pic

total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(torch.cuda.get_device_name(0), f"{total:.0f} Go")
try:
    pic = essai(False)
    RECALCUL = pic > 0.85 * total
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    pic, RECALCUL = float("nan"), True
if RECALCUL:
    pic = essai(True)
print(f"pic {pic:.1f} Go sur {total:.0f} | recalcul des activations : {RECALCUL}")
OPTS = f"--set model.recalcul_activations={str(RECALCUL).lower()}"

## 4. Entraînement, test 2022, fin de la session

In [ ]:
import shlex
def choisir_poids(src):
    """Le Drive garde parfois deux fichiers du meme nom (grand_tunis_v4_dso : deux best.pt de 1,69 Go,
    vus le 31/10) ; Colab peut n'en montrer qu'un, au hasard. On retient celui dont l'epoque est celle
    du test evalue (eval_pcfno3d/best_wcont, sinon best), meme s'il n'y en a qu'un : jamais un poids
    pris au hasard."""
    cands = sorted(src.glob("best*.pt"))
    if not cands:
        return src / "best.pt"
    attendu = next((json.loads(f.read_text()).get("epoque") for d in ("best_wcont", "best")
                    if (f := src / "eval_pcfno3d" / d / "test_summary.json").exists()), None)
    import torch
    epoques = {}
    for c in cands:
        ck = torch.load(str(c), map_location="cpu", mmap=True, weights_only=False)
        epoques[c] = ck.get("epoch", -2) + 1
        del ck
    print("poids candidats :", {c.name: e for c, e in epoques.items()}, "| epoque evaluee :", attendu)
    ok = [c for c in cands if epoques[c] == attendu] if attendu is not None else cands
    if not ok:
        raise RuntimeError(f"aucun poids de {src} n'a l'epoque evaluee ({attendu}) : verifie sur le Drive")
    return max(ok, key=lambda c: c.stat().st_mtime)          # a epoque egale : le plus recent

import shlex
from google.colab import runtime
import traceback
RUNS_GT = GT / "runs"
DECISION = RUNS_GT / "decision_v4.json"
livre = json.loads(DECISION.read_text())["modele_livre"] if DECISION.exists() else "grand_tunis_v2_long"
INIT = shlex.quote(str(choisir_poids(RUNS_GT / livre)))
RUN = NAT / "runs" / "national_1km_v2"
print("depart :", INIT)
try:
    # choix de l'epoque sur DSO1, DSO2, DSO3 de la validation, passe ici aussi : le carnet marche
    # alors avec une archive de code anterieure a ce reglage dans configs/national_1km_colab.yaml
    SEL = shlex.quote('train.select_criteres=["dso1_rmse","dso2_masse","dso3_rugosite"]')
    lancer("entrainer", f"python -u -m pcfno3d -c national_1km_colab {OPTS} --set {SEL} --set train.sync_last=true --set train.init_from={INIT} train")
    if not (RUN / "eval_pcfno3d" / "best_wcont" / "test_summary.json").exists():
        lancer("evaluer", f"python -u -m pcfno3d -c national_1km_colab {OPTS} evaluate")
    # DSO6 (fiabilite) sans reentrainement : ensemble par masquage, calibration sur la VALIDATION,
    # couverture 80 % et correlation incertitude / erreur sur le test (lu une fois)
    CAL = RUN / "fiabilite_pcfno3d" / "best_wcont" / "calibration_validation.json"
    D6 = RUN / "eval_pcfno3d" / "best_wcont" / "dso6_modele_livre.json"
    if not CAL.exists():
        lancer("calibrer_dso6", f"python -u -m pcfno3d -c national_1km_colab {OPTS} calibrate-reliability --out {CAL}")
    if not D6.exists():
        try:
            lancer("dso6", f"python -u -m pcfno3d -c national_1km_colab {OPTS} dso6 --calibration {CAL} --out {D6}")
        except RuntimeError as exc:                     # code 3 = NON CONFORME : verdict ecrit
            if "(code 3)" not in str(exc):
                raise
            print(exc)
    # validation sur les VRAIES mesures : 12 aeroports METAR (chacun retire de l'entree quand il
    # est juge) et radiosondages de Tunis-Carthage ; test 2022
    if (G / "obs").exists() and not (RUN / "validation_reelle" / "test" / "bilan.json").exists():
        lancer("validation_reelle", f"python -u scripts/valider_grand_tunis_reel.py -c national_1km_colab "
                                    f"--obs {G / 'obs'} --jeu test")
    for f in sorted(RUN.rglob("test_summary.json")):
        print(f); print(f.read_text()[:4000])
    for f in sorted(RUN.rglob("test_par_niveau_synoptique.csv")):
        print(pd.read_csv(f).round(3).to_string())
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()